# Khôi phục validation H3 sau khi Vertex bị chặn

Notebook này dùng **prompt validation cũ (giới hạn 8192 byte)**. Nó giữ response Qwen/Vertex đã thành công và gọi **DeepSeek-V4-Flash/FPT chỉ cho các request lỗi**. Lần chạy đã lưu có 2.714 request bị CONSUMER_SUSPENDED và 295 request bị lỗi 429; phải retry cả hai nhóm để validation hoàn tất. Bốn bản ghi ReVeal trùng prompt dùng chung request.

**Kết quả là exploratory, trộn model và trộn đời contract giữa validation/test. Không báo cáo điểm này như benchmark một backbone theo contract 24k.** Notebook ghi provider/model thực tế cho từng response và giữ bản gốc trong recovery/original/.

Thứ tự: kiểm kê → gọi thử 1 request → gọi phần còn lại → gộp và kích hoạt recovery → quay lại full-pipeline.ipynb, chạy lại từ Bước 1/2. Không chạy lại CPU fit; bước 1 sẽ dùng lại stage cũ khi marker recovery đã kích hoạt.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')
if not ROOT.exists():
    ROOT = Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))

from grace_h3v2.recovery import inventory, export_failure_inventory, retry_failed, finalize

STAGE_ID = '5fc21bd7cafff9846aff'
WORKERS = 4
print('Root:', ROOT)
print('Stage:', STAGE_ID)


Root: /mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble
Stage: 5fc21bd7cafff9846aff


## 1. Kiểm kê request lỗi

Bảng đọc các artifact validation đã có và xuất danh sách mẫu lỗi vào recovery/failure_inventory.jsonl; không gọi API. failed_unique_requests là số request cần retry, có thể thấp hơn failed_queries nếu prompt trùng.


In [2]:
report = inventory(ROOT, STAGE_ID)
display(pd.DataFrame(report['runs']))
print('Provider mới:', report['provider'], '| Model:', report['model'])
print('Nhóm lỗi:', {r['run']: r['error_categories'] for r in report['runs']})
print('Danh sách mẫu lỗi:')
display(pd.DataFrame(export_failure_inventory(ROOT, STAGE_ID)))


,run,queries,successful_vertex,failed_queries,failed_unique_requests,saved_journal_rows,error_categories
0,runs/devign/validation/baseline_whitening/f79c...,2724,1886,838,838,0,"{'rate_limited_429': 295, 'consumer_suspended'..."
1,runs/reveal/validation/baseline_whitening/27ee...,2196,25,2171,2167,0,{'consumer_suspended': 2171}


Provider mới: FPT AI Factory | Model: DeepSeek-V4-Flash
Nhóm lỗi: {'runs/devign/validation/baseline_whitening/f79c31718971e534': {'rate_limited_429': 295, 'consumer_suspended': 543}, 'runs/reveal/validation/baseline_whitening/27ee63ea9ec23f93': {'consumer_suspended': 2171}}
Danh sách mẫu lỗi:


,run,rows,file
0,runs/devign/validation/baseline_whitening/f79c...,838,runs/devign/validation/baseline_whitening/f79c...
1,runs/reveal/validation/baseline_whitening/27ee...,2171,runs/reveal/validation/baseline_whitening/27ee...


## 2. Gọi thử đúng một request FPT

Nếu tài khoản/model FPT chưa dùng được, dừng ở đây. Request thành công được ghi ngay vào journal và cache; chạy lại cell này sẽ chọn request kế tiếp.


In [5]:
smoke = retry_failed(ROOT, STAGE_ID, limit=1, workers=1)
print(smoke)
if smoke['errors']:
    raise RuntimeError('Request FPT đầu tiên thất bại; xem recovery/requests trong run tương ứng trước khi chạy hàng loạt.')


FPT retry requests: 1 (stage 5fc21bd7cafff9846aff)
FPT retry 1/1; ok=1, errors=0
{'attempted': 1, 'ok': 1, 'errors': 0, 'remaining': 3004}


## 3. Retry toàn bộ request lỗi còn lại

Có thể ngắt và chạy lại cell. Những request FPT đã thành công được lấy từ journal/cache, không gọi lại. Nếu còn errors hoặc remaining, xử lý nguyên nhân rồi chạy lại cell này.


In [6]:
result = retry_failed(ROOT, STAGE_ID, limit=0, workers=WORKERS)
print(result)
if result['remaining']:
    raise RuntimeError(f"Vẫn còn {result['remaining']} request chưa thành công; chạy lại cell sau khi xử lý lỗi.")


FPT retry requests: 3004 (stage 5fc21bd7cafff9846aff)
FPT retry 25/3004; ok=25, errors=0
FPT retry 50/3004; ok=50, errors=0
FPT retry 75/3004; ok=75, errors=0
FPT retry 100/3004; ok=100, errors=0
FPT retry 125/3004; ok=125, errors=0
FPT retry 150/3004; ok=150, errors=0
FPT retry 175/3004; ok=175, errors=0
FPT retry 200/3004; ok=200, errors=0
FPT retry 225/3004; ok=225, errors=0
FPT retry 250/3004; ok=250, errors=0
FPT retry 275/3004; ok=275, errors=0
FPT retry 300/3004; ok=300, errors=0
FPT retry 325/3004; ok=325, errors=0
FPT retry 350/3004; ok=350, errors=0
FPT retry 375/3004; ok=375, errors=0
FPT retry 400/3004; ok=400, errors=0
FPT retry 425/3004; ok=425, errors=0
FPT retry 450/3004; ok=450, errors=0
FPT retry 475/3004; ok=475, errors=0
FPT retry 500/3004; ok=500, errors=0
FPT retry 525/3004; ok=525, errors=0
FPT retry 550/3004; ok=550, errors=0
FPT retry 575/3004; ok=575, errors=0
FPT retry 600/3004; ok=600, errors=0
FPT retry 625/3004; ok=625, errors=0
FPT retry 650/3004; ok=650,

## 4. Gộp response và quay lại full pipeline

Cell này chỉ chạy khi mọi request lỗi đã có response FPT thành công. Nó giữ ba file gốc trong recovery/original/, tạo lại predictions/summary, rồi bật marker recovery. Sau đó khởi động lại kernel riêng của full-pipeline.ipynb để nạp mã workflow mới, rồi chạy từ Bước 1 hoặc Bước 2. Bước validation sẽ nhận status=complete, mixed_model_recovery=True.


In [7]:
marker = finalize(ROOT, STAGE_ID)
display(pd.DataFrame(marker['runs']))
print('Recovery active:', ROOT / 'runs/h3v2/mixed_model_recovery.json')
print('Quay lại notebooks/full-pipeline.ipynb. Đây là kết quả exploratory trộn Qwen/Vertex và DeepSeek/FPT.')


,run,providers,predictions_sha256
0,runs/devign/validation/baseline_whitening/f79c...,"{'Vertex AI': 1886, 'FPT AI Factory': 838}",2bf7205d1ef20c32c2bd24dfe57280b1aff4ac1b1a24ce...
1,runs/reveal/validation/baseline_whitening/27ee...,"{'FPT AI Factory': 2171, 'Vertex AI': 25}",03499f2cf242c2ea561b4b8e23a27b7ec55f67795f0840...


Recovery active: /mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble/runs/h3v2/mixed_model_recovery.json
Quay lại notebooks/full-pipeline.ipynb. Đây là kết quả exploratory trộn Qwen/Vertex và DeepSeek/FPT.
